In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("bronze_traffic_events")

display(bronze_df)

event_id,road_name,vehicle_count,avg_speed,weather,accident_status,latitude,longitude,timestamp,ingestion_time,source_system
1,Vaishali Nagar,499,17,Sunny,false,26.9407,75.7578,2026-08-02 11:16:06,2026-08-02T07:37:13.130Z,Python Traffic Simulator
10,C Scheme,486,61,Sunny,false,26.9071,75.7375,2026-08-02 11:17:49,2026-08-02T07:37:13.130Z,Python Traffic Simulator
100,C Scheme,638,27,Rain,false,26.8546,75.7903,2026-08-02 11:34:58,2026-08-02T07:37:13.130Z,Python Traffic Simulator
101,C Scheme,416,40,Cloudy,false,26.8769,75.8428,2026-08-02 11:35:09,2026-08-02T07:37:13.130Z,Python Traffic Simulator
102,MI Road,554,20,Sunny,false,26.8592,75.75,2026-08-02 11:35:21,2026-08-02T07:37:13.130Z,Python Traffic Simulator
103,Vaishali Nagar,136,22,Sunny,false,26.857,75.728,2026-08-02 11:35:32,2026-08-02T07:37:13.130Z,Python Traffic Simulator
104,Vaishali Nagar,502,46,Rain,false,26.9481,75.8376,2026-08-02 11:35:44,2026-08-02T07:37:13.130Z,Python Traffic Simulator
105,Ajmer Road,444,82,Sunny,true,26.9062,75.7949,2026-08-02 11:35:55,2026-08-02T07:37:13.130Z,Python Traffic Simulator
106,C Scheme,175,63,Rain,false,26.9375,75.7568,2026-08-02 11:36:07,2026-08-02T07:37:13.130Z,Python Traffic Simulator
107,Tonk Road,153,60,Rain,false,26.8941,75.7394,2026-08-02 11:36:19,2026-08-02T07:37:13.130Z,Python Traffic Simulator


In [0]:
silver_df = bronze_df.dropDuplicates()

In [0]:
silver_df = silver_df.filter(col("road_name").isNotNull())

In [0]:
silver_df = silver_df.filter(col("vehicle_count").isNotNull())

In [0]:
silver_df = silver_df.filter(col("avg_speed").isNotNull())

In [0]:
silver_df = silver_df.withColumn(
    "road_name",
    initcap(trim(col("road_name")))
)

In [0]:
silver_df = silver_df.filter(col("avg_speed") >= 0)

In [0]:
silver_df = silver_df.filter(col("vehicle_count") >= 0)

In [0]:
silver_df = silver_df.withColumn(
    "event_timestamp",
    to_timestamp(col("timestamp"))
)

In [0]:
silver_df = silver_df.withColumn(
    "processed_time",
    current_timestamp()
)

In [0]:
display(silver_df)

event_id,road_name,vehicle_count,avg_speed,weather,accident_status,latitude,longitude,timestamp,ingestion_time,source_system,event_timestamp,processed_time
35,Mi Road,320,32,Rain,false,26.9419,75.7391,2026-08-02 11:22:33,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:22:33.000Z,2026-08-02T09:59:11.141Z
42,Tonk Road,615,37,Fog,false,26.8934,75.7978,2026-08-02 11:23:53,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:23:53.000Z,2026-08-02T09:59:11.141Z
5,C Scheme,626,40,Fog,false,26.9421,75.8476,2026-08-02 11:16:52,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:16:52.000Z,2026-08-02T09:59:11.141Z
50,Mi Road,152,61,Sunny,false,26.8534,75.7506,2026-08-02 11:25:25,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:25:25.000Z,2026-08-02T09:59:11.141Z
60,Mi Road,334,54,Fog,true,26.8619,75.7379,2026-08-02 11:27:19,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:27:19.000Z,2026-08-02T09:59:11.141Z
64,C Scheme,136,48,Sunny,false,26.9173,75.8194,2026-08-02 11:28:05,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:28:05.000Z,2026-08-02T09:59:11.141Z
79,Tonk Road,288,86,Sunny,false,26.9268,75.7311,2026-08-02 11:30:57,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:30:57.000Z,2026-08-02T09:59:11.141Z
104,Vaishali Nagar,502,46,Rain,false,26.9481,75.8376,2026-08-02 11:35:44,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:35:44.000Z,2026-08-02T09:59:11.141Z
106,C Scheme,175,63,Rain,false,26.9375,75.7568,2026-08-02 11:36:07,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:36:07.000Z,2026-08-02T09:59:11.141Z
116,Mi Road,574,39,Cloudy,false,26.8956,75.7697,2026-08-02 11:38:02,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:38:02.000Z,2026-08-02T09:59:11.141Z


In [0]:
silver_path = "s3://smart-traffic-analytics/silver/traffic_events"

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver_traffic_cleaned
USING DELTA
LOCATION '{silver_path}'
""")

DataFrame[]

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM silver_traffic_cleaned
    """)
)

event_id,road_name,vehicle_count,avg_speed,weather,accident_status,latitude,longitude,timestamp,ingestion_time,source_system,event_timestamp,processed_time
35,Mi Road,320,32,Rain,false,26.9419,75.7391,2026-08-02 11:22:33,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:22:33.000Z,2026-08-02T09:59:13.112Z
42,Tonk Road,615,37,Fog,false,26.8934,75.7978,2026-08-02 11:23:53,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:23:53.000Z,2026-08-02T09:59:13.112Z
5,C Scheme,626,40,Fog,false,26.9421,75.8476,2026-08-02 11:16:52,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:16:52.000Z,2026-08-02T09:59:13.112Z
50,Mi Road,152,61,Sunny,false,26.8534,75.7506,2026-08-02 11:25:25,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:25:25.000Z,2026-08-02T09:59:13.112Z
60,Mi Road,334,54,Fog,true,26.8619,75.7379,2026-08-02 11:27:19,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:27:19.000Z,2026-08-02T09:59:13.112Z
64,C Scheme,136,48,Sunny,false,26.9173,75.8194,2026-08-02 11:28:05,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:28:05.000Z,2026-08-02T09:59:13.112Z
79,Tonk Road,288,86,Sunny,false,26.9268,75.7311,2026-08-02 11:30:57,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:30:57.000Z,2026-08-02T09:59:13.112Z
104,Vaishali Nagar,502,46,Rain,false,26.9481,75.8376,2026-08-02 11:35:44,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:35:44.000Z,2026-08-02T09:59:13.112Z
106,C Scheme,175,63,Rain,false,26.9375,75.7568,2026-08-02 11:36:07,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:36:07.000Z,2026-08-02T09:59:13.112Z
116,Mi Road,574,39,Cloudy,false,26.8956,75.7697,2026-08-02 11:38:02,2026-08-02T07:37:13.130Z,Python Traffic Simulator,2026-08-02T11:38:02.000Z,2026-08-02T09:59:13.112Z


In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM silver_traffic_cleaned
""").show()

+-------------+
|total_records|
+-------------+
|          268|
+-------------+

